In [1]:
from pathlib import Path
results_path = Path('output/results/CW_1x5_Dynamic_Complete/20251212-182251')
results_path

WindowsPath('output/results/CW_1x5_Dynamic_Complete/20251212-182251')

In [2]:
from src.reporting import load_results_from_path, parse_results_dict
results = load_results_from_path(results_path)

results_proposed_method = parse_results_dict(results['decision_tree-xgboost-dt-double'], 'double-GB')
results_proposed_method

c:\Users\ericv\code\stacked-new\.venv\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\ericv\code\stacked-new\.venv\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\ericv\code\stacked-new\.venv\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\ericv\code\stacked-new\.venv\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_p

{'151': {'train':     hybrid_n_correct_total  hybrid_n_correct_easy  hybrid_n_correct_hard  \
  0                      137                     71                     66   
  1                      124                     69                     55   
  2                      111                     51                     60   
  3                      131                     71                     60   
  4                      125                     64                     61   
  5                      127                     62                     65   
  6                      116                     41                     75   
  7                      111                     55                     56   
  8                      101                     60                     41   
  9                      116                     66                     50   
  10                     113                     76                     37   
  11                     111                    

In [3]:
results_without_reject = parse_results_dict(results['decision_tree-xgboost-dt-binary'], 'binary')
results_without_reject

{'151': {'train':     hybrid_n_correct_total  hybrid_n_correct_easy  hybrid_n_correct_hard  \
  0                      185                     71                    114   
  1                      186                     69                    117   
  2                      186                     51                    135   
  3                      185                     71                    114   
  4                      184                     64                    120   
  5                      186                     62                    124   
  6                      187                     41                    146   
  7                      186                     55                    131   
  8                      185                     60                    125   
  9                      185                     66                    119   
  10                     186                     76                    110   
  11                     185                    

In [12]:
import pandas as pd
from pathlib import Path

# Update this to the folder containing your 50 text files
data_dir = results_path / '891'

dfs = []
for i in range(50):
    file_path = data_dir / f"{i}-decision_tree-xgboost-dt-test-double.txt"
    if file_path.exists():
        dfs.append(pd.read_csv(file_path))
    else:
        print(f"Warning: File not found: {file_path.name}")

if dfs:
    # Combine all 50 runs into one DataFrame
    df = pd.concat(dfs, ignore_index=True)

    # A pattern is rejected if both graders equal 1
    df['rejected'] = (df['y_grader'] == 1) & (df['y_grader2'] == 1)

    # Group by the true class to see rejection stats
    rejection_stats = df.groupby('y_truth').agg(
        total_patterns=('rejected', 'count'),
        rejected_count=('rejected', 'sum')
    )
    
    # Calculate the percentage
    rejection_stats['rejection_rate (%)'] = (
        rejection_stats['rejected_count'] / rejection_stats['total_patterns'] * 100
    ).round(2)

    display(rejection_stats)
else:
    print("No files were loaded. Please check your data_dir path.")

,total_patterns,rejected_count,rejection_rate (%)
y_truth,,,
0,1091670,262137,24.01
1,176730,120300,68.07


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# Update this to the folder containing your 50 text files
data_dir = results_path / '572'

dfs = []
for i in range(50):
    file_path = data_dir / f"{i}-decision_tree-xgboost-dt-test-double.txt"
    if file_path.exists():
        dfs.append(pd.read_csv(file_path))

if dfs:
    df = pd.concat(dfs, ignore_index=True)
    
    # 1. Define predictions without reject (GB-Grader priority)
    df['y_pred_no_reject'] = np.where(df['y_grader'] == 0, df['y_glass'], df['y_black'])
    
    # 2. Define rejection mask (Both Graders = 1)
    df['rejected'] = (df['y_grader'] == 1) & (df['y_grader2'] == 1)
    
    results = []
    classes = sorted(df['y_truth'].unique())
    
    for c in classes:
        # --- NO REJECT METRICS ---
        tp_nr = len(df[(df['y_truth'] == c) & (df['y_pred_no_reject'] == c)])
        fp_nr = len(df[(df['y_truth'] != c) & (df['y_pred_no_reject'] == c)])
        fn_nr = len(df[(df['y_truth'] == c) & (df['y_pred_no_reject'] != c)])
        tn_nr = len(df[(df['y_truth'] != c) & (df['y_pred_no_reject'] != c)])
        
        prec_nr = tp_nr / (tp_nr + fp_nr) if (tp_nr + fp_nr) > 0 else 0.0
        rec_nr = tp_nr / (tp_nr + fn_nr) if (tp_nr + fn_nr) > 0 else 0.0
        spec_nr = tn_nr / (tn_nr + fp_nr) if (tn_nr + fp_nr) > 0 else 0.0
        
        # --- WITH REJECT METRICS ---
        # Filter to only the patterns the framework accepted
        df_acc = df[~df['rejected']]
        tp_wr = len(df_acc[(df_acc['y_truth'] == c) & (df_acc['y_pred_no_reject'] == c)])
        fp_wr = len(df_acc[(df_acc['y_truth'] != c) & (df_acc['y_pred_no_reject'] == c)])
        fn_wr = len(df_acc[(df_acc['y_truth'] == c) & (df_acc['y_pred_no_reject'] != c)])
        tn_wr = len(df_acc[(df_acc['y_truth'] != c) & (df_acc['y_pred_no_reject'] != c)])
        
        prec_wr = tp_wr / (tp_wr + fp_wr) if (tp_wr + fp_wr) > 0 else 0.0
        rec_wr = tp_wr / (tp_wr + fn_wr) if (tp_wr + fn_wr) > 0 else 0.0
        spec_wr = tn_wr / (tn_wr + fp_wr) if (tn_wr + fp_wr) > 0 else 0.0
        
        # Contextual Stats
        total_c = len(df[df['y_truth'] == c])
        rej_c = len(df[(df['y_truth'] == c) & df['rejected']])
        rej_rate = rej_c / total_c * 100 if total_c > 0 else 0.0
        
        results.append({
            'Class': c,
            'Total Patterns': total_c,
            'Rejected': rej_c,
            'Reject Rate (%)': round(rej_rate, 2),
            'Prec (No Rej)': round(prec_nr * 100, 2),
            'Prec (Reject)': round(prec_wr * 100, 2),
            'Rec (No Rej)': round(rec_nr * 100, 2),
            'Rec (Reject)': round(rec_wr * 100, 2),
            'Spec (No Rej)': round(spec_nr * 100, 2),
            'Spec (Reject)': round(spec_wr * 100, 2),
            'Accepted TP': tp_wr,
            'Accepted FP': fp_wr,
            'Accepted FN': fn_wr
        })
        
    res_df = pd.DataFrame(results)
    display(res_df)
else:
    print("No files were loaded. Please check your data_dir path.")

,Class,Total Patterns,Rejected,Reject Rate (%),Prec (No Rej),Prec (Reject),Rec (No Rej),Rec (Reject),Spec (No Rej),Spec (Reject),Accepted TP,Accepted FP,Accepted FN
0,0,32995,2238,6.78,97.63,98.96,99.42,99.96,27.73,1.23,30744,322,13
1,1,1100,774,70.36,61.24,23.53,27.73,1.23,99.42,99.96,4,13,322


In [7]:
import pandas as pd
from pathlib import Path

# Update this to the folder containing the Taiwanese Bankruptcy results
data_dir = results_path / '43'

partitions_with_zero_accepted = 0
total_partitions = 0
class_to_check = 1  # 1 is the minority class for bankruptcy

for i in range(50):
    file_path = data_dir / f"{i}-decision_tree-xgboost-dt-test-double.txt"
    
    if file_path.exists():
        total_partitions += 1
        df = pd.read_csv(file_path)
        
        # A pattern is rejected if both graders equal 1
        rejected_mask = (df['y_grader'] == 1) & (df['y_grader2'] == 1)
        
        # Filter to only the ACCEPTED patterns
        df_accepted = df[~rejected_mask]
        
        # Count how many accepted patterns belong to Class 1
        accepted_class_count = len(df_accepted[df_accepted['y_truth'] == class_to_check])
        
        # If the reject option wiped out the entire class for this fold
        if accepted_class_count == 0:
            partitions_with_zero_accepted += 1

if total_partitions > 0:
    print(f"Total partitions analyzed: {total_partitions}")
    print(f"Partitions with ZERO accepted patterns for Class {class_to_check}: {partitions_with_zero_accepted}")
else:
    print("No files were loaded. Please check your data_dir path.")

Total partitions analyzed: 50
Partitions with ZERO accepted patterns for Class 1: 0


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# Update this to the folder containing the Taiwanese Bankruptcy results
data_dir = results_path / '572'

partitions_with_zero_predictions = 0
total_partitions = 0
class_to_check = 1  # 1 is the minority class for bankruptcy

# Assuming standard DIFFICULTY mapping from earlier
DIFFICULTY_EASY = 0
DIFFICULTY_HARD = 1
DIFFICULTY_VERY_HARD = 2

for i in range(50):
    file_path = data_dir / f"{i}-decision_tree-xgboost-dt-test-double.txt"
    
    if file_path.exists():
        total_partitions += 1
        df = pd.read_csv(file_path)
        
        # Determine predictions without reject
        df['y_pred_no_reject'] = np.where(df['y_grader'] == 0, df['y_glass'], df['y_black'])
        
        # A pattern is rejected if both graders equal 1
        rejected_mask = (df['y_grader'] == 1) & (df['y_grader2'] == 1)
        
        # Filter to only the ACCEPTED patterns
        df_accepted = df[~rejected_mask]
        
        # Count how many times the model PREDICTED Class 1 in the accepted pool
        accepted_predictions_count = len(df_accepted[df_accepted['y_pred_no_reject'] == class_to_check])
        
        # If the model made ZERO predictions for Class 1 in this fold
        if accepted_predictions_count == 0:
            partitions_with_zero_predictions += 1

if total_partitions > 0:
    print(f"Total partitions analyzed: {total_partitions}")
    print(f"Partitions with ZERO Class {class_to_check} PREDICTIONS: {partitions_with_zero_predictions}")
else:
    print("No files were loaded. Please check your data_dir path.")

Total partitions analyzed: 50
Partitions with ZERO Class 1 PREDICTIONS: 35
